# SK-RD4AD — Data Augmentation Campaign (Colab, resumable batch)
Setup and dataset copy run **once**, then a loop iterates over a list of configs (baseline + ablations). Each run gets its own folder on Drive.

**Resumable:** if you re-run the notebook, runs already completed (marked with a `DONE` file) are **skipped** — the baseline is not recomputed. Add configs to the list anytime: only new ones are executed. A failure in one run does not stop the others.


In [ ]:
from google.colab import drive
import os, shutil, glob, json, subprocess, time
from datetime import datetime

drive.mount('/content/drive')

BRANCH = "feature/ablatable-augmentation"
if not os.path.isdir('/content/sk-rd4ad'):
    !git clone --branch {BRANCH} https://github.com/emanuelepietrocometti/sk-rd4ad.git
%cd /content/sk-rd4ad

!pip install -q --extra-index-url https://download.pytorch.org/whl/cu130 \
    torch>=2.1.0 torchvision>=0.16.0 numpy pandas scipy imageio matplotlib \
    opencv-python opencv-contrib-python Pillow scikit-image scikit-learn \
    fastprogress geomloss tqdm


# Campaign configuration
`HPARAMS` = the optimized configuration from ch. 5, **fixed** for the whole campaign. `AUG_CONFIGS` = the list of augmentation files to try; `SEEDS` = the list of seeds. Runs are the product `AUG_CONFIGS x SEEDS`, each in its own folder.


In [ ]:
# === DATASET / DRIVE ===
CLASS_NAME = "reda"          # custom textile class (ch. 5 dataset with noise in training)
DRIVE_BASE_PATH    = "/content/drive/MyDrive/Tesi/results_SKRD4AD_augmentation"
DRIVE_DATASET_PATH = f"/content/drive/MyDrive/Tesi/MVTec/{CLASS_NAME}"
LOCAL_DATASET_PATH = f"./mvtec/{CLASS_NAME}"

# === RUN LIST ===
# Add/remove configs freely: already-completed runs are skipped on re-run.
AUG_CONFIGS = [
    "configs/aug_off.json",          # <-- BASELINE
    "configs/aug_legacy.json",
    "configs/oat_affine.json",
    "configs/oat_blur.json",
    "configs/oat_brightness_contrast.json",
    "configs/oat_dynamic_crop.json",
    "configs/oat_equalize.json",
    "configs/oat_grayscale.json",
    "configs/oat_hflip.json",
    "configs/oat_hue.json",
    "configs/oat_speckle_high.json",
    "configs/oat_speckle_low.json",
    "configs/oat_vflip.json",
    "configs/combined_geometric.json",
    "configs/combined_candidate.json",
]
SEEDS = [42]   # fixed seed for the whole campaign

# === HPARAMS: OPTIMIZED configuration from ch. 5 (Table 4.2.4), FIXED ===
HPARAMS = {
    "learning_rate": 0.0005898, "batch_size": 4, "res": 1,
    "layerloss": 0, "rate": 0.0, "L2": 0, "net": "wide_res50",
    "cut": 1,          # no-op in main.py (see NOTES.md)
    "seg": 1, "vis": 0, "print_epoch": 50, "epochs": 200,
}

# Guard: config files must be present on the branch
missing = [c for c in AUG_CONFIGS if not os.path.exists(c)]
if missing:
    raise FileNotFoundError(f"Configs not found (did you push configs/ to branch '{BRANCH}'?): {missing}")

RUNS = [(cfg, s) for cfg in AUG_CONFIGS for s in SEEDS]
print(f"{len(RUNS)} runs queued ({len(AUG_CONFIGS)} configs x {len(SEEDS)} seeds)")


# Dataset preparation (once)
Copy the dataset to fast local storage. If already present in this session, skip.


In [ ]:
if not (os.path.isdir(LOCAL_DATASET_PATH) and os.listdir(LOCAL_DATASET_PATH)):
    print(f"Copying {CLASS_NAME} from Drive to local storage...")
    if os.path.exists(LOCAL_DATASET_PATH):
        shutil.rmtree(LOCAL_DATASET_PATH)
    shutil.copytree(DRIVE_DATASET_PATH, LOCAL_DATASET_PATH)
    print("Dataset copied.")
else:
    print("Dataset already present locally, skipping copy.")
os.makedirs(DRIVE_BASE_PATH, exist_ok=True)


# Resumable batch runner
For each `(config, seed)`: skip if `DONE` exists; otherwise train, evaluate and sync to Drive, writing the `DONE` marker at the end of the run. Partial (interrupted) runs are cleaned up and redone. A failure in one run does not stop the others.


In [ ]:
LOCAL_CKPT, LOCAL_RES = "./checkpoints", "./results"

def _clean(*dirs):
    for d in dirs:
        if os.path.exists(d): shutil.rmtree(d)
        os.makedirs(d, exist_ok=True)

def _sync(local_dir, drive_dir):
    if not os.path.isdir(local_dir): return
    os.makedirs(drive_dir, exist_ok=True)
    shutil.copytree(local_dir, drive_dir, dirs_exist_ok=True)

def run_one(aug_config, seed):
    tag      = os.path.splitext(os.path.basename(aug_config))[0]
    run_name = f"aug_{tag}_{CLASS_NAME}_seed{seed}"
    proj     = f"skrd4ad_{run_name}"
    drun     = os.path.join(DRIVE_BASE_PATH, run_name)
    dckpt, dres = os.path.join(drun, "checkpoints"), os.path.join(drun, "results")
    done_marker = os.path.join(drun, "DONE")

    if os.path.exists(done_marker):
        print(f"[skip] {run_name} (already completed)"); return "skipped"
    # partial/interrupted run -> clean up and redo
    if os.path.exists(drun):
        print(f"[redo] {run_name}: incomplete folder, cleaning up")
        shutil.rmtree(drun)

    with open(aug_config) as f: aug_cfg_content = json.load(f)
    _clean(LOCAL_CKPT, LOCAL_RES)
    os.makedirs(dckpt, exist_ok=True); os.makedirs(dres, exist_ok=True)
    with open(os.path.join(drun, "run_config.json"), "w") as f:
        json.dump({"run_name": run_name, "class_": CLASS_NAME, "seed": seed,
                   "project_name": proj, "dataset": DRIVE_DATASET_PATH,
                   "started_at": datetime.now().isoformat(timespec="seconds"),
                   "hparams": HPARAMS, "aug_config_path": aug_config,
                   "aug_config": aug_cfg_content}, f, indent=2)

    print(f"\n===== TRAIN {run_name} =====")
    train_cmd = ["python", "main.py",
        "--class_", CLASS_NAME, "--data_path", "./mvtec/",
        "--ckpt_path", LOCAL_CKPT + "/", "--img_path", LOCAL_RES + "/",
        "--project_name", proj, "--aug-config", aug_config, "--seed", str(seed),
        "--net", HPARAMS["net"], "--L2", str(HPARAMS["L2"]), "--res", str(HPARAMS["res"]),
        "--rate", str(HPARAMS["rate"]), "--batch_size", str(HPARAMS["batch_size"]),
        "--layerloss", str(HPARAMS["layerloss"]), "--seg", str(HPARAMS["seg"]),
        "--vis", str(HPARAMS["vis"]), "--print_epoch", str(HPARAMS["print_epoch"]),
        "--learning_rate", str(HPARAMS["learning_rate"]), "--epochs", str(HPARAMS["epochs"]),
        "--cut", str(HPARAMS["cut"])]
    if subprocess.run(train_cmd).returncode != 0:
        raise RuntimeError("training failed")

    ckpts = glob.glob(os.path.join(LOCAL_CKPT, "*.pth"))
    if not ckpts:
        raise FileNotFoundError("no checkpoint produced")
    latest = max(ckpts, key=os.path.getmtime)
    eval_img = os.path.join(LOCAL_RES, "eval_report"); os.makedirs(eval_img, exist_ok=True)
    print(f"===== EVAL {run_name} ({os.path.basename(latest)}) =====")
    subprocess.run(["python", "eval.py", "--class_", CLASS_NAME, "--data_path", "./mvtec/",
                    "--checkpoint_path", latest, "--img_path", eval_img + "/"])

    _sync(LOCAL_CKPT, dckpt); _sync(LOCAL_RES, dres)
    cfgp = os.path.join(drun, "run_config.json")
    c = json.load(open(cfgp)); c["finished_at"] = datetime.now().isoformat(timespec="seconds")
    json.dump(c, open(cfgp, "w"), indent=2)
    open(done_marker, "w").write(datetime.now().isoformat())
    print(f"[ok] {run_name} synced to Drive")
    return "done"

summary = {"done": [], "skipped": [], "failed": []}
for i, (aug_config, seed) in enumerate(RUNS, 1):
    print(f"\n########## RUN {i}/{len(RUNS)} ##########")
    try:
        status = run_one(aug_config, seed)
        summary[status].append((aug_config, seed))
    except Exception as e:
        print(f"[FAIL] {aug_config} seed={seed}: {e}")
        summary["failed"].append((aug_config, seed))

print("\n===== SUMMARY =====")
for k in ("done", "skipped", "failed"):
    print(f"{k}: {len(summary[k])} -> {[f'{os.path.basename(c)}#{s}' for c,s in summary[k]]}")


# Campaign status
Scan Drive and show which runs are complete and which are missing.


In [ ]:
print(f"Campaign: {DRIVE_BASE_PATH}\n")
for aug_config in AUG_CONFIGS:
    tag = os.path.splitext(os.path.basename(aug_config))[0]
    for seed in SEEDS:
        run_name = f"aug_{tag}_{CLASS_NAME}_seed{seed}"
        done = os.path.exists(os.path.join(DRIVE_BASE_PATH, run_name, "DONE"))
        print(f"  [{'x' if done else ' '}] {run_name}")
